In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import logging
import time

In [2]:
# Set up logging
# This configuration is for all modules that are in the same Python process, so it will affect all logging calls in this notebook
# To ensure all files use the same logging configuration, we can create a main.py file that sets up logging and then import the logger from there in all other files. 
logging.basicConfig(level=logging.INFO,
                    # filename='autoencoder_training.log',
                    # filemode='w',
                    format='%(levelname)s: %(message)s',
                    force=True) # Force reconfiguration to ensure settings are applied

In [4]:
logger = logging.getLogger(__name__) # Create a logger for the current file
logger.setLevel(logging.INFO) # Set the logging level to INFO for this logger
logger.info(f"Logger name: {logger.name} and level: {logging.getLevelName(logger.level)}") # Log the logger's name and level to verify configuration

INFO: Logger name: __main__ and level: INFO


In [9]:
time_variant_df = pd.read_csv('../datasets/all_features_all_data.csv')

if time_variant_df.empty:
    logger.warning("The loaded DataFrame is empty. Please check the file path and contents.")
else:
    logger.info(f"Data loaded successfully with shape: {time_variant_df.shape}")
    logger.info(f'DataFrame columns: {time_variant_df.columns.tolist()}')

INFO: Data loaded successfully with shape: (79152, 193)
INFO: DataFrame columns: ['city', 'state', 'zip', 'latitude_x', 'longitude_x', 'time', 'us_aqi', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_speed_100m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'cloud_cover', 'road_distance_m', 'road_impact_score', 'facility_count_nearby', 'facility_impact_score', 'overall_spatial_impact_score', 'month', 'month_sin', 'month_cos', 'day', 'hour', 'hour_sin', 'hour_cos', 'day_of_week', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year', 'is_weekend', 'wind_direction_10m_sin', 'wind_direction_10m_cos', 'wind_direction_100m_sin', 'wind_direction_100m_cos', 'latitude_y', 'longitude_y', 'us_aqi_past_1', 'us_aqi_past_2', 'us_aqi_past_3', 'us_aqi_past_4', 'us_aqi_past_5', 'us_aqi_past_6', 'us_aqi_

In [10]:
# Delete NaN values in the DataFrame
time_variant_df.dropna(inplace=True)
logger.info(f"Data cleaned successfully with shape: {time_variant_df.shape}")


INFO: Data cleaned successfully with shape: (76824, 193)


In [11]:
time_variant_df.drop(columns=['latitude_y', 'longitude_y', 'city', 'state', 'month', 'day', 'hour', 'day_of_week', 'day_of_year'], inplace=True)
logger.info(f"Unnecessary columns dropped. Remaining columns: {time_variant_df.columns.tolist()}")

INFO: Unnecessary columns dropped. Remaining columns: ['zip', 'latitude_x', 'longitude_x', 'time', 'us_aqi', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_speed_100m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'cloud_cover', 'road_distance_m', 'road_impact_score', 'facility_count_nearby', 'facility_impact_score', 'overall_spatial_impact_score', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend', 'wind_direction_10m_sin', 'wind_direction_10m_cos', 'wind_direction_100m_sin', 'wind_direction_100m_cos', 'us_aqi_past_1', 'us_aqi_past_2', 'us_aqi_past_3', 'us_aqi_past_4', 'us_aqi_past_5', 'us_aqi_past_6', 'us_aqi_past_7', 'us_aqi_past_8', 'us_aqi_past_9', 'us_aqi_past_10', 'us_aqi_past_11', 'us_aqi_past_12', 'us_aqi_past_13', 'us_aqi_past

In [12]:
remove_cols = ['zip', 'time', 'us_aqi']
feature_cols = [col for col in time_variant_df.columns if col not in remove_cols]
logger.info(f"Feature columns: {feature_cols}")

INFO: Feature columns: ['latitude_x', 'longitude_x', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'uv_index_clear_sky', 'uv_index', 'dust', 'aerosol_optical_depth', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_speed_100m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'cloud_cover', 'road_distance_m', 'road_impact_score', 'facility_count_nearby', 'facility_impact_score', 'overall_spatial_impact_score', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend', 'wind_direction_10m_sin', 'wind_direction_10m_cos', 'wind_direction_100m_sin', 'wind_direction_100m_cos', 'us_aqi_past_1', 'us_aqi_past_2', 'us_aqi_past_3', 'us_aqi_past_4', 'us_aqi_past_5', 'us_aqi_past_6', 'us_aqi_past_7', 'us_aqi_past_8', 'us_aqi_past_9', 'us_aqi_past_10', 'us_aqi_past_11', 'us_aqi_past_12', 'us_aqi_past_13', 'us_aqi_past_14', 'us_aqi_past_15', 'us_aqi_past_16', 'us_aqi_past_1

In [ ]:
# Group by 'zip', sort by 'time', and create sequences
train_splits = []
test_splits = []

for zip_code, group in time_variant_df.groupby('zip'):
    group = group.sort_values('time').reset_index(drop=True)
    split_index = int(len(group) * 0.8)
    train_splits.append(group.iloc[:split_index])
    test_splits.append(group.iloc[split_index:])

train_df = pd.concat(train_splits, axis=0).reset_index(drop=True) # Combine all training splits into a single DataFrame
test_df = pd.concat(test_splits, axis=0).reset_index(drop=True)

X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values
logger.info(f"Training split shape: {X_train.shape}")
logger.info(f"Testing split shape: {X_test.shape}")
logger.info(f'First training split sample:\n{train_df[feature_cols].iloc[0]}')

y_train = train_df['us_aqi'].values
y_test = test_df['us_aqi'].values
logger.info(f"Training labels shape: {y_train.shape}")
logger.info(f"Testing labels shape: {y_test.shape}")

INFO: Training split shape: (61401, 181)
INFO: Testing split shape: (15423, 181)
INFO: First training split sample:
latitude_x                         29.756350
longitude_x                       -95.365380
pm10                               31.300000
pm2_5                              30.900000
carbon_monoxide                   636.000000
                                     ...    
wind_direction_10m_cos_past_20      0.990257
wind_direction_10m_cos_past_21      0.997308
wind_direction_10m_cos_past_22      0.999885
wind_direction_10m_cos_past_23      0.999889
wind_direction_10m_cos_past_24      1.000000
Name: 0, Length: 181, dtype: float64
INFO: Training labels shape: (61401,)
INFO: Testing labels shape: (15423,)


In [ ]:
# Scale the features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
logger.info("Feature scaling completed successfully.")
logger.info(f'Feature range after scaling: min={X_train_scaled.min()}, max={X_train_scaled.max()}')
logger.info(f'Feature range after scaling: min={X_test_scaled.min()}, max={X_test_scaled.max()}')

scaler2 = StandardScaler()
X_train_scaled_std = scaler2.fit_transform(X_train)
X_test_scaled_std = scaler2.transform(X_train)
logger.info("Feature scaling completed successfully.")
logger.info(f'Feature range after scaling: min={X_train_scaled_std.min()}, max={X_train_scaled_std.max()}')
logger.info(f'Feature range after scaling: min={X_test_scaled_std.min()}, max={X_test_scaled_std.max()}')



INFO: Feature scaling completed successfully.
INFO: Feature range after scaling: min=0.0, max=1.0000000000000002
INFO: Feature range after scaling: min=-0.006923837784371909, max=1.088235294117647


In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
logger.info("Data converted to PyTorch tensors successfully.")

INFO: Data converted to PyTorch tensors successfully.


In [ ]:
# Convert to PyTorch tensors
X_train_tensor_std = torch.tensor(X_train_scaled_std, dtype=torch.float32)
X_test_tensor_std = torch.tensor(X_test_scaled_std, dtype=torch.float32)
logger.info("Data converted to PyTorch tensors successfully.")

In [ ]:
class EarlyStopping:
  def __init__(self, patience=5):
    self.patience = patience
    self.counter = 0;
    self.best_loss = float("inf")
    self.best_state = None
  
  def step(self, model, val_loss):
    if(val_loss < self.best_loss):
      self.best_loss = val_loss
      self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
      self.counter = 0
      return False
    else:
      self.counter += 1
      return self.counter >= self.patience

class TimeVariantAutoencoder(nn.Module):
    def __init__(self, input_dim, scaler=MinMaxScaler, hidden_layers=[128, 64]):
        super(TimeVariantAutoencoder, self).__init__()
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_layers[:-1]:
            encoder_layers.append(nn.Linear(prev_dim, hidden_dim))
            encoder_layers.append(nn.ReLU())
            prev_dim = hidden_dim
        encoder_layers.append(nn.Linear(prev_dim, hidden_layers[-1])) # Final layer to latent space
        self.encoder = nn.Sequential(*encoder_layers)

        # Decoder
        decoder_layers = []
        prev_dim = hidden_layers[-1]
        for hidden_dim in reversed(hidden_layers[:-1]):
            decoder_layers.append(nn.Linear(prev_dim, hidden_dim))
            decoder_layers.append(nn.ReLU())
            prev_dim = hidden_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers, nn.Sigmoid()) if scaler == MinMaxScaler else nn.Sequential(*decoder_layers)

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return latent, reconstructed

    def compile(self, optimizer, loss_fn):
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.earlystopper = EarlyStopping()

    def fit(self, train_loader, val_loader, num_epochs=50):
        self.train() # Set the model to training mode
        for epoch in range(num_epochs):

            epoch_loss = 0.0
            for (X_batch, )in train_loader:
                _, reconstructed = self.forward(X_batch)

                self.optimizer.zero_grad()
                loss = self.loss_fn(reconstructed, X_batch)
                loss.backward()
                self.optimizer.step()
                epoch_loss += loss.item() * X_batch.size(0) # Accumulate loss
            
            avg_loss = epoch_loss / len(train_loader.dataset) # Average loss for the epoch

            val_loss = self.validate(val_loader)

            if (epoch + 1) % 10 == 0 or epoch == 0: # Log every 10 epochs and the first epoch
                logger.info(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.8f}, , Val Loss: {val_loss:.8f}')

            if(self.earlystopper.step(self, val_loss)):
              logger.info(f"Early stopping at epoch {epoch+1}")
              break
              
        return self.earlystopper.best_loss

    def predict(self, X):
        self.eval() # Set the model to evaluation mode
        with torch.no_grad():
            _, reconstructed = self.forward(X)
        return reconstructed

    def validate(self, test):
        self.eval() # Set the model to evaluation mode
        val_loss = 0.0
        with torch.no_grad():
            for (X_batch, )in test:
                _, reconstructed = self.forward(X_batch)
                loss = self.loss_fn(reconstructed, X_batch)
                val_loss += loss.item() * X_batch.size(0) # len(X_batch) is the number of samples in the batch, so we multiply by it to get the total loss for that batch
        avg_loss = val_loss / len(test.dataset) # Average loss for the test set
        #logger.info(f'Val Loss: {avg_loss:.8f}')
        return avg_loss

In [ ]:
hidden_layers = [
    [64],
    [32],
    # [16],
    # [64, 16],
    # [128, 32],
    # [64, 32, 8]
]

In [ ]:
class AETrainer:
    def __init__(self):
        self.results = []
    
    def train_and_evaluate(self, X, X_test, scaler, hidden_layers):
        for hidden_layer_config in hidden_layers:
            logger.info(f"Training autoencoder with hidden layers: {hidden_layer_config}")
            model = TimeVariantAutoencoder(input_dim=X.shape[1], scaler=scaler, hidden_layers=hidden_layer_config)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            loss_fn = nn.MSELoss()

            model.compile(optimizer, loss_fn)

            train_dataset = TensorDataset(X)
            train_loader = DataLoader(train_dataset, batch_size=1048, shuffle=True)

            val_dataset = TensorDataset(X_test)
            val_loader = DataLoader(val_dataset, batch_size=1048, shuffle=True)

            start_time = time.time()
            best_loss = model.fit(train_loader, val_loader, num_epochs=50)
            end_time = time.time()
            logger.info(f"Training completed in {end_time - start_time:.4f} seconds")

            self.results.append({
                'hidden_layers': hidden_layer_config,
                'training_time': end_time - start_time,
                'test_loss': best_loss
            })


In [ ]:
trainer_minmax = AETrainer()
trainer_minmax.train_and_evaluate(X_train_tensor, X_test_tensor, MinMaxScaler, hidden_layers)
results_minmax = trainer_minmax.results

# trainer_std = AETrainer()
# trainer_std.train_and_evaluate(X_train_tensor_std, X_test_tensor_std, StandardScaler, hidden_layers)
# results_std = trainer_std.results

In [ ]:
logger.info(f'Using MinMax:')
for result in results_minmax:
    logger.info(f"Hidden Layers: {result['hidden_layers']}\tTraining Time: {result['training_time']:.4f} seconds\tTest Loss: {result['test_loss']:.8f}")

# logger.info(f'Using StandardScaler:')
# for result in results_std:
#     logger.info(f"Hidden Layers: {result['hidden_layers']}\tTraining Time: {result['training_time']:.4f} seconds\tTest Loss: {result['test_loss']:.8f}")